In [1]:
import requests
import time
from datetime import datetime, timedelta
from typing import Dict, List, Optional
import json
from collections import defaultdict
from tqdm import tqdm
import pandas as pd
from datasets import load_dataset


/home/local/QCRI/fdeniz/anaconda3/envs/vllm_v100/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
"""
Augment Natural Questions dataset with Wikipedia pageview data.
This script:
1. Loads Natural Questions from HuggingFace
2. Extracts Wikipedia entities from the data
3. Fetches pageview statistics from Wikipedia API
4. Saves augmented dataset with popularity metrics
"""

import requests
import time
from datetime import datetime, timedelta
from typing import Dict, List, Optional
import json
from collections import defaultdict
from tqdm import tqdm
import pandas as pd

# Install required packages:
# pip install datasets wikipedia-api requests tqdm pandas

from datasets import load_dataset


class WikipediaPageviewFetcher:
    """Fetch Wikipedia pageview statistics."""
    
    def __init__(self, user_agent="Research/1.0"):
        self.base_url = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
        self.headers = {"User-Agent": user_agent}
        self.cache = {}
    
    def get_pageviews(
        self,
        title: str,
        start_date: str = None,
        end_date: str = None,
        project: str = "en.wikipedia"
    ) -> Optional[int]:
        """
        Get average daily pageviews for a Wikipedia article.
        
        Args:
            title: Wikipedia article title (with underscores, e.g., "Albert_Einstein")
            start_date: Start date in YYYYMMDD format (default: 30 days ago)
            end_date: End date in YYYYMMDD format (default: yesterday)
            project: Wikipedia project (default: en.wikipedia)
        
        Returns:
            Average daily pageviews or None if error
        """
        # Clean title
        title = title.replace(" ", "_")
        
        # Check cache
        cache_key = f"{project}:{title}"
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        # Set default dates (last 30 days)
        if not start_date or not end_date:
            end = datetime.now() - timedelta(days=1)
            start = end - timedelta(days=30)
            start_date = start.strftime("%Y%m%d")
            end_date = end.strftime("%Y%m%d")
        
        # Build URL
        url = f"{self.base_url}/{project}/all-access/user/{title}/daily/{start_date}/{end_date}"
        
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                items = data.get("items", [])
                
                if items:
                    # Calculate average daily views
                    total_views = sum(item["views"] for item in items)
                    avg_views = total_views // len(items)
                    self.cache[cache_key] = avg_views
                    return avg_views
            
            elif response.status_code == 404:
                # Article not found
                self.cache[cache_key] = 0
                return 0
            
            else:
                print(f"Error fetching {title}: Status {response.status_code}")
                return None
                
        except Exception as e:
            print(f"Exception fetching {title}: {e}")
            return None
        
        # Rate limiting
        time.sleep(0.1)
        return None


def extract_wikipedia_title(url: str) -> Optional[str]:
    """Extract Wikipedia article title from URL."""
    if not url or "wikipedia.org/wiki/" not in url:
        return None
    
    try:
        # Extract title from URL
        title = url.split("/wiki/")[-1]
        # Remove any anchor links
        title = title.split("#")[0]
        return title
    except:
        return None


def process_natural_questions(
    num_samples: int = 1000,
    split: str = "validation"
) -> pd.DataFrame:
    """
    Process Natural Questions dataset and augment with pageview data.
    
    Args:
        num_samples: Number of samples to process (set to None for all)
        split: Dataset split to use ('train' or 'validation')
    
    Returns:
        DataFrame with augmented data
    """
    print("Loading Natural Questions dataset...")
    dataset = load_dataset("google-research-datasets/natural_questions", split=split)
    
    if num_samples:
        dataset = dataset.select(range(min(num_samples, len(dataset))))
    
    print(f"Processing {len(dataset)} samples...")
    
    fetcher = WikipediaPageviewFetcher()
    results = []
    
    for idx, example in enumerate(tqdm(dataset)):
        # Extract basic info
        question = example["question"]["text"]
        document_url = example["document"]["url"]
        
        # Extract Wikipedia title
        wiki_title = extract_wikipedia_title(document_url)
        
        if not wiki_title:
            continue
        
        # Get annotations (answers)
        annotations = example.get("annotations", {})
        short_answers = []
        long_answers = []
        
        if annotations:
            # Extract short answers
            for annotation in annotations.get("short_answers", []):
                if annotation:
                    for ans in annotation:
                        start = ans.get("start_token", -1)
                        end = ans.get("end_token", -1)
                        if start >= 0 and end >= 0:
                            short_answers.append({"start": start, "end": end})
            
            # Extract long answers
            for annotation in annotations.get("long_answer", []):
                if annotation:
                    start = annotation.get("start_token", -1)
                    end = annotation.get("end_token", -1)
                    if start >= 0 and end >= 0:
                        long_answers.append({"start": start, "end": end})
        
        # Fetch pageviews
        pageviews = fetcher.get_pageviews(wiki_title)
        
        results.append({
            "question_id": example.get("id", idx),
            "question": question,
            "wiki_title": wiki_title,
            "wiki_url": document_url,
            "pageviews_avg_daily": pageviews,
            "has_short_answer": len(short_answers) > 0,
            "has_long_answer": len(long_answers) > 0,
            "short_answers": short_answers,
            "long_answers": long_answers
        })
        
        # Progress update every 100 samples
        if (idx + 1) % 100 == 0:
            print(f"Processed {idx + 1} samples. Cache size: {len(fetcher.cache)}")
    
    return pd.DataFrame(results)


def analyze_popularity_distribution(df: pd.DataFrame):
    """Analyze the popularity distribution of the dataset."""
    print("\n" + "="*60)
    print("POPULARITY ANALYSIS")
    print("="*60)
    
    valid_views = df[df["pageviews_avg_daily"].notna()]
    
    print(f"\nTotal samples: {len(df)}")
    print(f"Samples with pageview data: {len(valid_views)}")
    print(f"Samples without pageview data: {len(df) - len(valid_views)}")
    
    if len(valid_views) > 0:
        print(f"\nPageview Statistics:")
        print(f"  Mean: {valid_views['pageviews_avg_daily'].mean():.2f}")
        print(f"  Median: {valid_views['pageviews_avg_daily'].median():.2f}")
        print(f"  Min: {valid_views['pageviews_avg_daily'].min():.2f}")
        print(f"  Max: {valid_views['pageviews_avg_daily'].max():.2f}")
        
        # Percentiles
        print(f"\nPercentiles:")
        for p in [10, 25, 50, 75, 90, 95, 99]:
            val = valid_views['pageviews_avg_daily'].quantile(p/100)
            print(f"  {p}th percentile: {val:.2f}")
        
        # Categorize by popularity
        print(f"\nPopularity Categories:")
        print(f"  Very popular (>10k views/day): {(valid_views['pageviews_avg_daily'] > 10000).sum()}")
        print(f"  Popular (1k-10k views/day): {((valid_views['pageviews_avg_daily'] >= 1000) & (valid_views['pageviews_avg_daily'] <= 10000)).sum()}")
        print(f"  Moderate (100-1k views/day): {((valid_views['pageviews_avg_daily'] >= 100) & (valid_views['pageviews_avg_daily'] < 1000)).sum()}")
        print(f"  Low (<100 views/day): {(valid_views['pageviews_avg_daily'] < 100).sum()}")


def save_dataset(df: pd.DataFrame, output_path: str = "nq_with_pageviews.jsonl"):
    """Save the augmented dataset."""
    df.to_json(output_path, orient="records", lines=True)
    print(f"\nDataset saved to: {output_path}")
    
    # Also save a CSV for easy viewing
    csv_path = output_path.replace(".jsonl", ".csv")
    df.to_csv(csv_path, index=False)
    print(f"CSV version saved to: {csv_path}")


if __name__ == "__main__":
    # Process a sample of Natural Questions
    # For full dataset, set num_samples=None (will take many hours!)
    
    print("Starting Natural Questions augmentation with Wikipedia pageviews...")
    print("NOTE: Processing the full dataset will take many hours due to API rate limits.")
    print("Starting with 1000 samples for demonstration.\n")
    
    # Process dataset
    df = process_natural_questions(
        num_samples=1000,  # Change to None for full dataset
        split="train"
    )
    
    # Analyze popularity distribution
    analyze_popularity_distribution(df)
    
    # Save results
    save_dataset(df)
    
    # Show some examples
    print("\n" + "="*60)
    print("SAMPLE DATA")
    print("="*60)
    print(df[["question", "wiki_title", "pageviews_avg_daily"]].head(10).to_string())
    
    print("\nDone! You can now use this dataset for your research on knowledge popularity.")

Starting Natural Questions augmentation with Wikipedia pageviews...
NOTE: Processing the full dataset will take many hours due to API rate limits.
Starting with 1000 samples for demonstration.

Loading Natural Questions dataset...
Processing 1000 samples...


100%|██████████| 1000/1000 [00:19<00:00, 50.85it/s]



POPULARITY ANALYSIS


KeyError: 'pageviews_avg_daily'

## Google AI

In [ ]:
import requests
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
import time
from datetime import datetime, timedelta

# 1. SETUP: Define headers to be polite to Wikipedia API
# (Wikipedia requires a User-Agent identifying your bot/email)
HEADERS = {
    'User-Agent': 'ResearchBot/1.0 (your_email@example.com)'
}

def get_30_day_page_views(title):
    """
    Fetches the total page views for a Wikipedia article over the last 30 days.
    """
    if not title:
        return 0
    
    # Format title for API (Spaces become underscores)
    safe_title = title.replace(" ", "_")
    
    # Calculate dates
    end_date = datetime.now()
    start_date = end_date - timedelta(days=30)
    str_start = start_date.strftime("%Y%m%d")
    str_end = end_date.strftime("%Y%m%d")

    url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/all-agents/{safe_title}/daily/{str_start}/{str_end}"

    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code == 200:
            data = response.json()
            # Sum up the daily views
            total_views = sum([item['views'] for item in data.get('items', [])])
            return total_views
    except Exception:
        pass
    
    return 0

def main():
    print("Loading Natural Questions (Validation Set)...")
    # We load 'validation' because 'train' is 300k+ examples and takes very long to api-call
    dataset = load_dataset("google-research-datasets/natural_questions", split="validation")
    
    # Convert to pandas for easier manipulation
    df = pd.DataFrame(dataset)
    
    # For demonstration, let's take a sample of 100 to prove it works. 
    # COMMENT OUT the line below to run on the full dataset.
    df = df.head(100) 

    print(f"Processing {len(df)} questions...")

    # 2. EXTRACT TITLES AND GET VIEWS
    page_views = []
    
    # We use tqdm to show a progress bar
    for title in tqdm(df['document_title']):
        views = get_30_day_page_views(title)
        page_views.append(views)
        
        # Sleep slightly to avoid hitting API rate limits (100 req/sec is the limit, but be safe)
        time.sleep(0.05)

    # 3. AUGMENT DATASET
    df['page_views_30d'] = page_views

    # 4. SAVE
    output_file = "nq_augmented_with_popularity.csv"
    df.to_csv(output_file, index=False)
    print(f"Done! Saved to {output_file}")
    
    # Preview
    print(df[['question', 'document_title', 'page_views_30d']].head())

if __name__ == "__main__":
    main()